In [9]:
import sys
import json
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))
sys.path.append(str(Path().resolve().parents[1]))


from main.persona_types.Persona import *

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "key"

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4", temperature=0, seed=42)
shallow_persona_llm = llm.with_structured_output(PersonaShallow)
shallow_persona_list_llm = llm.with_structured_output(PersonaShallowList)
persona_llm = llm.with_structured_output(Persona)

In [12]:
from typing_extensions import TypedDict

class InterviewTurn(TypedDict):
    question: str
    time_seconds: int


class State(TypedDict):
  source_row: dict
  persona_shallow: PersonaShallow | None
  persona: Persona | None
  close_social_personas_shallow: list[PersonaShallow] | None
  close_social_personas: list[Persona] | None
  interview_transcript: list[InterviewTurn] | None


In [13]:
from langchain.messages import HumanMessage, SystemMessage


def convert_nemo_to_persona_shallow(state: State) -> dict:
  row = state["source_row"]
  persona_shallow: PersonaShallow = shallow_persona_llm.invoke([
            SystemMessage(
                content=(
                    "Convert the input row into the PersonaShallow schema. "
                    "Preserve directly specified equivalent fields. "
                    "For missing fields, infer semantically consistent values "
                    "from the provided data. Do not contradict explicit facts. "
                    "Prefer plausible completion over unnecessary invention."
                )
            ),
            HumanMessage(content=str(row)),
        ])

  return {
        "persona_shallow": persona_shallow,
    }

def expand_anchor_persona(state: State) -> dict:
  shallow_persona = state["persona_shallow"]
  interview_transcript = state["interview_transcript"]
  persona: Persona = persona_llm.invoke([
            SystemMessage(
                content=(
                    "Considering the shallow persona object and interview transcript of that persona, expand to a full Persona object."
                    "For persona_id, demographic, psych_traits, and social network you should be able to directly port the values from the input."
                    "Set close_social_circle to null."
                )
            ),
            HumanMessage(
                content="Persona Shallow:\n" + shallow_persona.model_dump_json(indent=2)
            ),
            HumanMessage(
                content="Interview Transcript:\n" + json.dumps(interview_transcript, indent=2)
            )
        ])

  return {
        "persona": persona,
    }

def create_close_social_personas_shallow(state: State) -> dict:
  persona = state["persona"]
  interview_transcript = state["interview_transcript"]
  response: PersonaShallowList = shallow_persona_list_llm.invoke([
            SystemMessage(
                content=(
                    "Considering the given persona and interview transcript with that persona, generate a list of 4 or 5 close social personas."
                    "Close social personas are personas that the persona represented by the anchor is likely to have regular contact or correspondence with."
                    "You may refer to the interview transcript to infer people likely to be close to this persona, the interviewee is the persona you are creating."
                    "Note that these will be shallow personas."
                )
            ),
            HumanMessage(
                content="Persona:\n" + persona.model_dump_json(indent=2)
            ),
            HumanMessage(
                content="Interview Transcript:\n" + json.dumps(interview_transcript, indent=2)
            )
        ])

  return {
        "close_social_personas_shallow": response.personas,
    }



def expand_close_social_personas(state: State) -> dict:
  close_social_persona_expanded_list: list[Persona] = []
  anchor_persona = state["persona"]
  close_social_personas = state["close_social_personas_shallow"]

  for close_social_persona in close_social_personas:
    persona: Persona = persona_llm.invoke([
              SystemMessage(
                  content=(
                      "Considering the shallow persona object expand to a full Persona object."
                      "For persona_id, demographic, psych_traits, and social network you should be able to directly port the values from the input."
                      "Do not contradict explicit facts. Prefer plausible completion over unnecessary invention."
                      "You can mirror any references in the input anchor persona hidden context in the output persona."
                  )
              ),
              HumanMessage(content=str(anchor_persona)),
              HumanMessage(content=str(close_social_persona)),
          ])
    close_social_persona_expanded_list.append(persona)

  return {
        "close_social_personas": close_social_persona_expanded_list,
    }





In [14]:
from langgraph.graph import StateGraph, START, END


agent_builder = StateGraph(State)

agent_builder.add_node("convert_shallow", convert_nemo_to_persona_shallow)
agent_builder.add_node("expand_anchor", expand_anchor_persona)
agent_builder.add_node("create_close_social_personas_shallow", create_close_social_personas_shallow)
agent_builder.add_node("expand_social_circle", expand_close_social_personas)

agent_builder.add_edge(START, "convert_shallow")
agent_builder.add_edge("convert_shallow", "expand_anchor")
agent_builder.add_edge("expand_anchor", "create_close_social_personas_shallow")
agent_builder.add_edge("create_close_social_personas_shallow", "expand_social_circle")
agent_builder.add_edge("expand_social_circle", END)

agent = agent_builder.compile()

In [16]:
# UPDATE !!!!!!!!!!
import json

with open("../data/alberti.json", "r") as f:
    persona_seed = json.load(f)

sample = [persona_seed]
print(sample)

[{'uuid': 'e7c0574639a244c8972c92aab9501035', 'professional_persona': 'Mary Alberti is a front-line food service specialist whose razor-sharp cash handling, inventory tracking, and POS mastery combine with a disciplined, routine-driven work ethic, enabling them to calmly resolve high-pressure customer issues and hit performance targets while eyeing a promotion to shift supervisor.', 'sports_persona': 'Mary Alberti fuels their fitness routine by clocking 3-5 mile runs around Lake Mendota with the Madison Runners Club, roots for the Wisconsin Badgers basketball team in the winter, cheers the Milwaukee Brewers in summer, and never misses a Green Bay Packers game on Sundays, balancing competitive spirit with disciplined consistency.', 'arts_persona': 'Mary Alberti finds creative inspiration in the lyrical storytelling of John Prine, the atmospheric indie folk of Bon Iver, and the classic cinematography of Wes Anderson, often attending local art walks and museum exhibits to unwind after a s

In [18]:
with open("../enrich/interview_transcripts/alberti_interview_transcript.json") as f:
    interview_ts = json.load(f)

print(interview_ts[0])

{'speaker': 'Interviewer', 'text': 'Some people save for big things, like a home, while others save for a rainy day. How about for you (in the last year)?'}


In [19]:
import json
import os

os.makedirs("expanded_personas", exist_ok=True)

anchor_persona_seeds = sample

for row in anchor_persona_seeds:

    initial_state = {
        "source_row": row,
        "persona_shallow": None,
        "persona": None,
        "close_social_personas_shallow": None,
        "close_social_personas": None,
        "interview_transcript": interview_ts
    }

    result = agent.invoke(initial_state)

    personas = [result["persona"]] + result["close_social_personas"]

    with open(f'expanded_personas/{personas[0].persona_id}.json', "w") as f:
        json.dump([p.model_dump() for p in personas], f, indent=2)

In [21]:
with open("./expanded_personas/personas.json") as f:
    personas = json.load(f)

print(personas[1])

{'persona_id': 'close_1_roommate_sara', 'demographics': {'name': 'Sara Jensen', 'age': 27, 'gender': 'Female', 'education': 'Some college', 'occupation': 'Medical receptionist', 'income_bracket': '$35k-$75k', 'location': 'Madison, WI 53717, USA', 'marital_status': 'Never married'}, 'psych_traits': {'openness': 0.58, 'conscientiousness': 0.82, 'extraversion': 0.48, 'agreeableness': 0.72, 'neuroticism': 0.39}, 'preferences_and_interests': {'health_and_wellness': 'Sara values practical, sustainable health habits over intense fitness routines. She tries to keep regular sleep, stay hydrated during busy front-desk shifts, and fit in walks, light workouts, or occasional yoga to offset time spent sitting and managing patients all day. She sees wellness as part of staying steady, patient, and dependable.', 'food': 'Her food habits are budget-aware and convenience-conscious but not careless. She likes easy weeknight meals, coffee, meal-prep basics, salads, soups, pasta dishes, and shareable groc

In [27]:
from langchain.messages import HumanMessage, SystemMessage

valid_prompt = """
Given a persona and their close social circle, give a rating for each member of the close social circle (1-7) based on the persona scoring how realistic and plausible it would be for the member to be part of the focal personas close circle.

Respond only with a list of scores.
"""


res = llm.invoke([
    SystemMessage(content=valid_prompt),
    HumanMessage(content="Focal Persona: " + json.dumps(personas[0], indent=2)),
    HumanMessage(content="Close Circle Personas: " + json.dumps(personas[1:], indent=2)),
])

print(res)

content='7\n7\n6\n3\n6' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 6686, 'total_tokens': 6698, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DORKnzSMZ5kcyweOOupAqvfh4zfiK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019d3565-2e4c-7282-b368-cdaded687731-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6686, 'output_tokens': 12, 'total_tokens': 6698, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [28]:
with open("./expanded_personas/validation_score.json", "w") as f:
    vals = res.content.split("\n")
    nums = [int(x) for x in vals if x.strip()]
    avg = sum(nums) / len(nums)

    
    data = {"personas_qa_score": avg, "personas_qa_all_scores": res.content} 
    json.dump(data, f, indent=2)